# Agentic Homogeneity Hypothesis

Several recent DDA researchs has a tendency to modelate defficulty based on the performance of pretrained RL agents. This proposal is build on the idea that any agent whould performe worst as more difficulte is the level, so there should be a correspondance between agents performance and levels intrinsic difficulty. Nonetheless, that relationship also requires some kind of homogeneity among solvers behavior, that we will call the Agentic Homogeneity Hypothesis, which remains unverifiyed or formaly caracterizate in the current bibliografy.

González-Duque et al. works from the hypothesis but, consciously or unconsciously, they adress to test it analysing convergence of their DDA method. The Bayesian-based Intelligent Trial-and-Error Algorithm (IT&E) use the performance of a priori agent to estimate another agent performance (posteriori) and use it as fit function for a custom evolutive algorithm. As part of their results they show that IT&E works better for certain priors-posteriori pairs, curiosly IT&E prove to be idempotent and symetric, i.e. predicting the performance from an identically behavior agent works just rigth and no trivial pairs that works well do it in both ways (it is no true for optimal pairing)*.

Extrapolating IT&E results further from it's levels adaptation methods and out of zelda-like game design, we can conclude that there should exist sufficent conditions to link priori and posteriori agents to set a good predictor. This experiment intend to corrovorate the Agentic Homogeneity Hypothesis in the SMB framework extending González-Duque et al. scope establishing the sufficent information to link priori and posteriori behaviors.


References:

* González-Duque et al., ‘Finding Game Levels with the Right Difficulty in a Few Trials through Intelligent Trial-and-Error’.
* Summerville et al., ‘The VGLC’.
* Lucas and Volz, ‘Tile Pattern KL-Divergence for Analysing and Evolving Game Levels’.

## Known difficulty regime

We extract the following list of SMB levels from The Video Games Level Corpues by Summerville et al. and classify them as table shows from Lucas and Volz KL-Divergence.

| Order | File Name | World | Stage | Level Type | Sub-Cluster |
| --- | --- | --- | --- | --- | --- |
| 1 | `mario-6-3.txt` | 6 | 3 | Athletic | 1A |
| 2 | `mario-3-3.txt` | 3 | 3 | Athletic | 1A |
| 3 | `mario-5-3.txt` | 5 | 3 | Athletic | 1B |
| 4 | `mario-1-3.txt` | 1 | 3 | Athletic | 1B |
| 5 | `mario-4-2.txt` | 4 | 2 | Underground | 2A |
| 6 | `mario-1-2.txt` | 1 | 2 | Underground | 2A |
| 7 | `mario-8-1.txt` | 8 | 1 | Overworld | 3A-1 |
| 8 | `mario-5-1.txt` | 5 | 1 | Overworld | 3A-1 |
| 9 | `mario-6-1.txt` | 6 | 1 | Overworld | 3A-2 |
| 10 | `mario-4-1.txt` | 4 | 1 | Overworld | 3A-2 |
| 11 | `mario-6-2.txt` | 6 | 2 | Overworld | 3B-1 |
| 12 | `mario-7-1.txt` | 7 | 1 | Overworld | 3B-1 |
| 13 | `mario-2-1.txt` | 2 | 1 | Overworld | 3B-1 |
| 14 | `mario-3-1.txt` | 3 | 1 | Overworld | 3B-2 |
| 15 | `mario-1-1.txt` | 1 | 1 | Overworld | 3B-2

Notice that levels are difficulty well ordered by human design, i.e. we can asumme that World and Stage labels constitute a criteria to dicide which level is more difficult.

Descarting the models training casuistry, the solver behavior is defined by 2 complex components: policy (models family, architecture and parameters) and trainning data. Because our objective is to stablish relation among the models perse we will limit the whole analysis to those 15 levels.


### Trainning-Test Subsetting

Because a single agent is define by it's policy and trainning set, it will allow us to stratify comparations among diferent agents, from completly diferent ones to similar ones. By similar I mean that share policy ($=Family,\space \not = architecture\space or\space parameters$) or tranning ($= Level\space Type\space mixture,\space \not = Subclusters$). 

## Proposed Model Roster & Taxonomy

To establish a rigorous, field-standard benchmark for Super Mario Bros (SMB), the player model suite should span the spectrum from **deterministic baselines** to **tabular/linear approximations** and **deep end-to-end policies**.

---

*Observation Contracts*: `features`, `tiles`, `rgb`

*Models*
The models are organized into four architectural tiers, tagged by **Paradigm**, **Observation Contract**, **Compute/Sample Requirement**, and **Interpretability**.

#### Tier 0: Control & Heuristic Baselines (Non-Learning)

*Purpose: Establish lower-bound floors, sanity-check environment physics/rewards, and provide deterministic speed benchmarks.*

| Model Name | Paradigm Tag | Obs Contract | Compute / Sample Req | Interpretability | Role in Investigation |
| --- | --- | --- | --- | --- | --- |
| **`random_seeded`** | `[Control: Stochastic]` | Any (Agnostic) | Negligible / Zero-shot | **High** (Uniform distribution) | Absolute lower bound; verifies action-space coverage (`Discrete(8)`). |
| **`right_runner`** | `[Control: Blind Heuristic]` | Any (Agnostic) | Negligible / Zero-shot | **Exact** (Static rule) | Tests pure forward momentum (`index 6: right + run`); measures how far Mario gets without jumping. |
| **`forward_jumper`** | `[Control: Blind Heuristic]` | Any (Agnostic) | Negligible / Zero-shot | **Exact** (Periodic counter) | Classic Mario AI benchmark baseline; alternates `right + run` (`6`) and `right + run + jump` (`7`) to clear basic gaps/pipes without perception. |
| **`reactive_avoider`** | `[Heuristic: Rule-Based]` | `features` or `tiles` | Low / Zero-shot | **Exact** (Explicit `if/else` thresholds) | Upper-bound non-ML control; inspects front tile/enemy proximity to trigger jump (`4` or `7`) and cooldowns. |

---

#### Tier 1: Tabular & Linear Function Approximation

*Purpose: High-interpretability learning baselines that isolate credit assignment and exploration dynamics without deep neural network overhead.*

| Model Name | Paradigm Tag | Obs Contract | Compute / Sample Req | Interpretability | Role in Investigation |
| --- | --- | --- | --- | --- | --- |
| **`tabular_q`** *(Implemented)* | `[RL: Off-Policy Value]` | `features` (discretized) | Low CPU / High episodes | **High** (Direct $Q(s,a)$ lookup) | Baseline temporal-difference control; exposes state-aliasing limits in continuous/large maps. |
| **`tabular_sarsa`** | `[RL: On-Policy Value]` | `features` (discretized) | Low CPU / High episodes | **High** (Direct $Q(s,a)$ lookup) | Contrasts with `tabular_q` on risk sensitivity—learns safer margins around pits due to $\epsilon$-greedy on-policy updates. |
| **`linear_sarsa_fa`** | `[RL: Linear Approx]` | `features` / `features_layers` | Low CPU / Moderate episodes | **High** (Inspectable feature weights $w_i$) | Bridges tabular and deep RL; feature weights directly reveal which state variables (e.g., pit distance, enemy velocity) drive jump decisions. |

---

#### Tier 2: Deep Value-Based Methods (Off-Policy)

*Purpose: Field-standard discrete control algorithms evaluating spatial generalization and Q-value overestimation bias.*

| Model Name | Paradigm Tag | Obs Contract | Compute / Sample Req | Interpretability | Role in Investigation |
| --- | --- | --- | --- | --- | --- |
| **`mlp_ddqn`** | `[Deep RL: Off-Policy Value]` | `features` / `tiles` | Moderate CPU/GPU / Replay Buffer | **Medium** (Saliency / Grad-CAM on inputs) | Isolates policy learning from visual perception; fast convergence on structured state data without CNN bottlenecks. |
| **`cnn_dqn`** *(e.g., `torch_dqn`)* | `[Deep RL: Off-Policy Value]` | `rgb` (frame-stacked) | High GPU / Large Replay Buffer | **Low-Medium** (Pixel saliency maps) | Standard Atari/SMB deep RL baseline; susceptible to Q-value overestimation and replay buffer imbalance on sparse rewards. |
| **`cnn_ddqn_dueling`** | `[Deep RL: Off-Policy Value]` | `rgb` (frame-stacked) | High GPU / Large Replay Buffer | **Medium** (Separates $V(s)$ and $A(s,a)$) | Field-standard upgrade to DQN; Dueling architecture explicitly shows *when* state danger matters ($V(s)$) vs. *which* action matters ($A(s,a)$). |

---

#### Tier 3: Deep Policy Gradient & Actor-Critic Methods (On-Policy)

*Purpose: Modern industry standard for stable, high-variance platformer navigation and direct stochastic policy optimization.*

| Model Name | Paradigm Tag | Obs Contract | Compute / Sample Req | Interpretability | Role in Investigation |
| --- | --- | --- | --- | --- | --- |
| **`a2c`** *(Advantage Actor-Critic)* | `[Deep RL: On-Policy Actor-Critic]` | `features` or `rgb` | Moderate GPU / Synchronous rollouts | **Medium** (Action probability distributions + Critic $V(s)$) | Lightweight policy-gradient baseline; measures variance and sample efficiency without trust-region clipping. |
| **`ppo`** *(Proximal Policy Opt.)* | `[Deep RL: On-Policy Actor-Critic]` | `features` and `rgb` (both variants) | Moderate-High GPU / Rollout horizon | **Medium** (Policy entropy, KL divergence, Critic $V(s)$) | **Gold-standard SMB benchmark**; clipped surrogate objective prevents catastrophic policy collapse when encountering new level obstacles. |

---

 **6-agent core benchmark**:

1. **`forward_jumper`** *(Non-ML baseline floor)*
2. **`reactive_avoider`** *(Rule-based feature ceiling)*
3. **`tabular_q`** *(Existing ML baseline)*
4. **`linear_sarsa_fa`** *(Interpretable linear function approximation)*
5. **`cnn_ddqn_dueling`** *(Best-in-class value-based visual RL)*
6. **`ppo` (`features` & `rgb`)** *(Best-in-class policy-gradient RL across both observation regimes)*